# Seq2Seq 모델 Q&A + Chatbot 만들기

1. qna 데이터셋을 찾아서 처리해서 준비한다.( 전처리 전반)
2. 인코더디코더seq2seq(인코더+디코더) 모델 만들기
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습
4. 챗봇을 만든다(모델 추론 +while문)

### Chatbot_data_for_Korean v1.0
- 제작자: songys(송영숙) (GitHub 사용자)
- 도메인: 일상 대화 (연애, 감정, 고민 등)
- 데이터 용량: 11876
- 목적: 한국어 챗봇 학습 및 자연어 생성 모델 연구
- 특징: 질문-답변(Q&A) 형태의 대화 데이터이며 감정 라벨(일상, 이별, 사랑)이 포함되어 있어 감정 기반 챗봇 학습에도 활용 가능

출처: https://github.com/songys/Chatbot_data

In [16]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
ds = df[['Q', 'A']]
ds

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


In [ ]:
# 사용한 라이브러리
import re
import pandas as pd
from konlpy.tag import Okt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
# 하이퍼 파라미터 설정, 전역변수 
BATCH_SIZE = 64         # 배치 크기
MAX_VOCAB_SIZE = 30000  # 최대 단어 집합의 크기
EMBEDDING_DIM = 100     # 단어 임베딩의 차원 수
LATENT_DIM = 512        # LSTM의 은닉 상태의 차원 수

In [7]:
# 형태소 분석기 초기화
okt = Okt()

# 토큰화 함수
def tokenizer(text):
    return okt.morphs(text)

# 반복 문자 정규화 함수 
# 예: "ㅋㅋㅋㅋ" -> "ㅋㅋ", "ㅎㅎㅎㅎ" -> "ㅎㅎ", "ㅠㅠㅠㅠ" -> "ㅠㅠ"
def normalize_repeats(text):
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ가-힣])\1{3,}', r'\1\1', text) 
    text = re.sub(r'([ㅋㅎㅠㅜ])\1{2,}', r'\1\1', text)
    text = re.sub(r'([!?.,~])\1{2,}', r'\1\1', text)
    
    # 줄바꿈, 탭 제거
    text = re.sub(r'[\r\n\t]+', ' ', text) 
    text = re.sub(r'\s+', ' ', text)
    return text

# 텍스트 정제 함수
# 공백 제거, 반복 문자 정규화
def clean_text(text):
    text = str(text).strip()
    text = normalize_repeats(text)
    return text

# 토큰 리스트를 다시 문장으로 합치는 함수
# 예: ["안녕", "하세", "요", "!"] -> "안녕하세요!"
def detokenize(tokens):
    sentence = " ".join(tokens)
    sentence = re.sub(r"\s+([?.!,~])", r"\1", sentence)
    sentence = re.sub(r"\(\s+", "(", sentence)
    sentence = re.sub(r"\s+\)", ")", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    return sentence

# 문장 하나 전처리
def preprocess_text(text):
    text = clean_text(text)
    tokens = tokenizer(text)
    return detokenize(tokens)


# DataFrame 전처리
def preprocess_data(df, q_col='Q', a_col='A'):
    df = df[[q_col, a_col]].copy()
    df = df.rename(columns={q_col: 'question', a_col: 'answer'})

    # 결측치 제거
    df = df.dropna(subset=['question', 'answer'])

    # 문자열화 + 공백 제거
    df['question'] = df['question'].astype(str).str.strip()
    df['answer'] = df['answer'].astype(str).str.strip()

    # 빈 문자열 제거
    df = df[(df['question'] != '') & (df['answer'] != '')].reset_index(drop=True)

    # 전처리
    df['question'] = df['question'].apply(preprocess_text)
    df['answer'] = df['answer'].apply(preprocess_text)

    return df[['question', 'answer']]


In [ ]:
# 전처리
preprocessed_df = preprocess_data(df, q_col='Q', a_col='A')

# 확인
print(preprocessed_df.head())

# 기존 코드 재사용을 위해 dict 리스트로 변환
preprocessed_ds = preprocessed_df.to_dict(orient='records')

print(preprocessed_ds[0])
print(len(preprocessed_ds))

# 저장
preprocessed_df.to_csv('./data/preprocessed_data.csv', index=False, encoding='utf-8-sig')

             question        answer
0              12시 땡!   하루 가 또 가네요.
1        1 지망 학교 떨어졌어    위로 해 드립니다.
2     3 박 4일 놀러 가고 싶다  여행 은 언제나 좋죠.
3  3 박 4일 정도 놀러 가고 싶다  여행 은 언제나 좋죠.
4             PPL 심하네   눈살 이 찌푸려지죠.
{'question': '12시 땡!', 'answer': '하루 가 또 가네요.'}
11823


In [ ]:
# 불러오기
preprocessed_ds = pd.read_csv('./data/preprocessed_data.csv')
preprocessed_ds = preprocessed_ds.to_dict(orient='records')

In [ ]:
# vocab 만들기
special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
counter = Counter()

# 질문/답변의 단어 추출해빈도수 계산 
for item in preprocessed_ds:
    question_tokens = item['question'].split()
    answer_tokens = item['answer'].split()

    counter.update(question_tokens)
    counter.update(answer_tokens)

vocab_words = [word for word, _ in counter.most_common(MAX_VOCAB_SIZE)]
vocab = special_tokens + vocab_words

word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# 문장을 정수 시퀀스로 변환
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    question_tokens = sample['question'].split()
    answer_tokens = sample['answer'].split()

    q_ids = sentence_to_sequence(question_tokens, word2idx)
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(answer_tokens, word2idx) + [word2idx['<EOS>']]

    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })

print(tokenized_data[0])

# padding
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    seq = seq[:max_len]
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)

print(tokenized_data[0])

{'question_ids': [4664, 7446], 'answer_ids': [1, 187, 5, 100, 2880, 2]}
{'question_ids': [4664, 7446, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer_ids': [1, 187, 5, 100, 2880, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [ ]:
class ChatDataset(Dataset):  
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        encoder_input = sample['question_ids']

        answer_ids = sample['answer_ids']
        decoder_input = answer_ids[:-1]
        decoder_target = answer_ids[1:]

        return {
            'encoder_input': torch.tensor(encoder_input, dtype=torch.long),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }

In [22]:
# 데이터 분할
train_data, val_data = train_test_split(tokenized_data, test_size=0.1, random_state=42)

train_dataset = ChatDataset(train_data)
val_dataset = ChatDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 문장을 정수 시퀀스
def sentence_to_sequence(tokens, word2idx):
    return [word2idx.get(token, word2idx['<UNK>']) for token in tokens]

tokenized_data = []

for sample in preprocessed_ds:
    question_tokens = sample['question'].split()
    answer_tokens = sample['answer'].split()

    # 질문 문장
    q_ids = sentence_to_sequence(question_tokens, word2idx)

    # 답변 문장
    a_ids = [word2idx['<SOS>']] + sentence_to_sequence(answer_tokens, word2idx) + [word2idx['<EOS>']]

    tokenized_data.append({
        'question_ids': q_ids,
        'answer_ids': a_ids
    })

print(tokenized_data[0])

{'question_ids': [4664, 7446], 'answer_ids': [1, 187, 5, 100, 2880, 2]}


In [11]:
# padding 처리
max_q_len = max(len(sample['question_ids']) for sample in tokenized_data)
max_a_len = max(len(sample['answer_ids']) for sample in tokenized_data)

def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

pad_idx = word2idx['<PAD>']
for sample in tokenized_data:
    sample['question_ids'] = pad_sequence(sample['question_ids'], max_q_len, pad_idx)
    sample['answer_ids'] = pad_sequence(sample['answer_ids'], max_a_len, pad_idx)
    
print((tokenized_data[0]))

{'question_ids': [4664, 7446, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'answer_ids': [1, 187, 5, 100, 2880, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [13]:
# Encoder 모델 구현
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [14]:
# Decoder 모델 구현
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, (h_s, c_s)

In [15]:
# Seq2Seq 모델 구현
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, (h_s, c_s) = self.decoder(target, h_s, c_s)
    return output

In [ ]:
vocab_size = len(word2idx)
# Encoder와 Decoder 모델 초기화 
# 파라미터 설정
encoder = Encoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>'],
    embedding_matrix=None
)

decoder = Decoder(
    vocab_size,
    EMBEDDING_DIM,
    LATENT_DIM,
    pad_idx=word2idx['<PAD>']
)

model = Seq2Seq(encoder, decoder)
model

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(13846, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(13846, 100, padding_idx=0)
    (lstm): LSTM(100, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=13846, bias=True)
  )
)

##### cpu

In [ ]:
# cpu
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):

    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for batch in train_loader:

        enc_inputs = batch['encoder_input']
        dec_inputs = batch['decoder_input']
        dec_targets = batch['decoder_target']

        optimizer.zero_grad()

        output = model(enc_inputs, dec_inputs)

        output = output.view(-1, output.size(-1))
        dec_targets = dec_targets.view(-1)

        loss = criterion(output, dec_targets)
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)

        train_loss += loss.detach().item()

        mask = dec_targets != 0
        correct = (preds == dec_targets) & mask

        train_correct += correct.sum().item()
        train_tokens += mask.sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # validation
    model.eval()
    val_loss, val_correct, val_tokens = 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:

            enc_inputs = batch['encoder_input']
            dec_inputs = batch['decoder_input']
            dec_targets = batch['decoder_target']

            output = model(enc_inputs, dec_inputs)

            output = output.view(-1, output.size(-1))
            dec_targets = dec_targets.view(-1)

            loss = criterion(output, dec_targets)

            preds = output.argmax(dim=-1)

            val_loss += loss.detach().item()

            mask = dec_targets != 0
            correct = (preds == dec_targets) & mask

            val_correct += correct.sum().item()
            val_tokens += mask.sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens

    val_losses.append(val_loss)
    val_accs.append(val_acc)
    # 3분 1에폭
    print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/10 TrainLoss=6.1807 TrainAcc=0.1805 ValLoss=5.4088 ValAcc=0.2102
Epoch 2/10 TrainLoss=5.2156 TrainAcc=0.2355 ValLoss=4.7431 ValAcc=0.2609
Epoch 3/10 TrainLoss=4.6134 TrainAcc=0.2785 ValLoss=4.1307 ValAcc=0.3120
Epoch 4/10 TrainLoss=4.0546 TrainAcc=0.3249 ValLoss=3.5763 ValAcc=0.3679
Epoch 5/10 TrainLoss=3.5506 TrainAcc=0.3790 ValLoss=3.0974 ValAcc=0.4357
Epoch 6/10 TrainLoss=3.1179 TrainAcc=0.4399 ValLoss=2.7162 ValAcc=0.5049
Epoch 7/10 TrainLoss=2.7619 TrainAcc=0.4978 ValLoss=2.4112 ValAcc=0.5597
Epoch 8/10 TrainLoss=2.4779 TrainAcc=0.5492 ValLoss=2.1870 ValAcc=0.6045
Epoch 9/10 TrainLoss=2.2524 TrainAcc=0.5915 ValLoss=2.0084 ValAcc=0.6382
Epoch 10/10 TrainLoss=2.0803 TrainAcc=0.6234 ValLoss=1.8749 ValAcc=0.6643


In [ ]:
# gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):

    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for batch in train_loader:

        enc_inputs = batch['encoder_input'].to(device)
        dec_inputs = batch['decoder_input'].to(device)
        dec_targets = batch['decoder_target'].to(device)

        optimizer.zero_grad()

        output = model(enc_inputs, dec_inputs)

        output = output.view(-1, output.size(-1))
        dec_targets = dec_targets.view(-1)

        loss = criterion(output, dec_targets)
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)

        train_loss += loss.detach().item()

        mask = dec_targets != 0
        correct = (preds == dec_targets) & mask

        train_correct += correct.sum().item()
        train_tokens += mask.sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens

    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # validation
    model.eval()
    val_loss, val_correct, val_tokens = 0, 0, 0

    with torch.no_grad():
        for batch in val_loader:

            enc_inputs = batch['encoder_input'].to(device)
            dec_inputs = batch['decoder_input'].to(device)
            dec_targets = batch['decoder_target'].to(device)

            output = model(enc_inputs, dec_inputs)

            output = output.view(-1, output.size(-1))
            dec_targets = dec_targets.view(-1)

            loss = criterion(output, dec_targets)

            preds = output.argmax(dim=-1)

            val_loss += loss.detach().item()

            mask = dec_targets != 0
            correct = (preds == dec_targets) & mask

            val_correct += correct.sum().item()
            val_tokens += mask.sum().item()

    val_loss /= len(val_loader)
    val_acc = val_correct / val_tokens

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Using device: cuda
Epoch 1/100 TrainLoss=3.3401 TrainAcc=0.4232 ValLoss=2.8407 ValAcc=0.4886
Epoch 2/100 TrainLoss=2.8452 TrainAcc=0.4867 ValLoss=2.4514 ValAcc=0.5514
Epoch 3/100 TrainLoss=2.4956 TrainAcc=0.5417 ValLoss=2.1541 ValAcc=0.6088
Epoch 4/100 TrainLoss=2.2300 TrainAcc=0.5909 ValLoss=1.9672 ValAcc=0.6466
Epoch 5/100 TrainLoss=2.0413 TrainAcc=0.6288 ValLoss=1.8146 ValAcc=0.6738
Epoch 6/100 TrainLoss=1.8782 TrainAcc=0.6581 ValLoss=1.6936 ValAcc=0.6957
Epoch 7/100 TrainLoss=1.7719 TrainAcc=0.6770 ValLoss=1.6267 ValAcc=0.7071
Epoch 8/100 TrainLoss=1.7247 TrainAcc=0.6855 ValLoss=1.5744 ValAcc=0.7156
Epoch 9/100 TrainLoss=1.6283 TrainAcc=0.7018 ValLoss=1.5056 ValAcc=0.7248
Epoch 10/100 TrainLoss=1.5644 TrainAcc=0.7109 ValLoss=1.4560 ValAcc=0.7303
Epoch 11/100 TrainLoss=1.5170 TrainAcc=0.7167 ValLoss=1.4183 ValAcc=0.7355
Epoch 12/100 TrainLoss=1.4775 TrainAcc=0.7202 ValLoss=1.3820 ValAcc=0.7397
Epoch 13/100 TrainLoss=1.4395 TrainAcc=0.7258 ValLoss=1.3438 ValAcc=0.7454
Epoch 14/100 Tr

In [ ]:
# 모델 저장
torch.save(model.state_dict(), 'seq2seq_chatbot.pth')

<All keys matched successfully>

### 시연

In [1]:
import pandas as pd
import torch

In [19]:
# 파일 불러오기
preprocessed_ds = pd.read_csv('./data/preprocessed_data.csv')
preprocessed_ds = preprocessed_ds.to_dict(orient='records')
# 모델 불러오기
model = Seq2Seq(encoder, decoder)
model.load_state_dict(torch.load('./data/seq2seq_chatbot.pth', map_location=torch.device('cpu')))

<All keys matched successfully>

In [ ]:
# 챗봇 구현
# 모델 추론 + while True: 반복문으로 사용자 입력 받기
def translate(input_seq, model, word2idx, idx2word, max_len=20):
    model.eval()

    device = next(model.parameters()).device

    encoder = model.encoder
    decoder = model.decoder

    input_seq = torch.tensor([input_seq], dtype=torch.long).to(device)

    with torch.no_grad():
        hidden, cell = encoder(input_seq)

    sos_index = word2idx['<SOS>']
    eos_index = word2idx['<EOS>']
    pad_index = word2idx['<PAD>']

    output_sentence = []

    target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output, (hidden, cell) = decoder(target_seq, hidden, cell)

            # 마지막 시점의 예측 토큰
            pred = output[:, -1, :].argmax(dim=-1).item()

            if pred == eos_index:
                break

            if pred != pad_index:
                word = idx2word.get(pred, '<UNK>')
                output_sentence.append(word)

            target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

    return ' '.join(output_sentence)

# 사용자 입력 
def sentence_to_ids(sentence):
    sentence = preprocess_text(sentence)   
    tokens = sentence.split()              

    ids = [word2idx.get(t, word2idx['<UNK>']) for t in tokens]

    ids = ids[:max_q_len]

    ids = pad_sequence(ids, max_q_len, word2idx['<PAD>'])

    return ids

# --------------------------------------
def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

def preprocess_text(text):
    text = clean_text(text)
    tokens = tokenizer(text)
    return detokenize(tokens)
def clean_text(text):
    text = str(text).strip()
    text = normalize_repeats(text)
    return text

In [21]:
# while 문으로 사용자 입력 받기

while True:
    print("챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력")
    user_input = input("사용자: ")
    
    if user_input.lower() == '종료':
        print("챗봇을 종료합니다.")
        break
    else:
        print(f"사용자 입력: {user_input}")

    input_ids = sentence_to_ids(user_input)
    response = translate(input_ids, model, word2idx, idx2word)
    print(f"챗봇: {response}")

챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력
사용자 입력: 오늘이 지났어
챗봇: 시간 이 흐르면 무 덤덤해질 거 예요.
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력
사용자 입력: 졸려
챗봇: 오늘 일찍 주무 세 요.
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력
사용자 입력: 개강이다
챗봇: 곧 방학 이 예요.
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력
사용자 입력: 간만에 쇼핑했어
챗봇: 득템 했길 바라요.
챗봇과 대화하려면 메시지를 입력하세요. 종료하려면 '종료' 입력
챗봇을 종료합니다.


In [77]:
for i in range(100):
    print("Q:", preprocessed_ds[i]['question'])
    print("A:", preprocessed_ds[i]['answer'])
    print('-' * 50)

Q: 12시 땡!
A: 하루 가 또 가네요.
--------------------------------------------------
Q: 1 지망 학교 떨어졌어
A: 위로 해 드립니다.
--------------------------------------------------
Q: 3 박 4일 놀러 가고 싶다
A: 여행 은 언제나 좋죠.
--------------------------------------------------
Q: 3 박 4일 정도 놀러 가고 싶다
A: 여행 은 언제나 좋죠.
--------------------------------------------------
Q: PPL 심하네
A: 눈살 이 찌푸려지죠.
--------------------------------------------------
Q: SD 카드 망가졌어
A: 다시 새로 사는 게 마음 편해요.
--------------------------------------------------
Q: SD 카드 안 돼
A: 다시 새로 사는 게 마음 편해요.
--------------------------------------------------
Q: SNS 맞팔 왜 안 하지 ㅠㅠ
A: 잘 모르고 있을 수도 있어요.
--------------------------------------------------
Q: SNS 시간 낭비 인 거 아는데 매일 하는 중
A: 시간 을 정 하고 해보세요.
--------------------------------------------------
Q: SNS 시간 낭비 인데 자꾸 보게 됨
A: 시간 을 정 하고 해보세요.
--------------------------------------------------
Q: SNS 보면 나 만 빼고 다 행복 해보여
A: 자랑 하는 자리 니까 요.
--------------------------------------------------
Q: 가끔 궁금해
A: 그 사람 도 그럴 거 예요.
----------